In [ ]:
%pip install transformers datasets peft accelerate evaluate scikit-learn

In [ ]:
import os
import glob
import pandas as pd

base_path = "../data"

def load_custom_dataset(base_path):
    data = []

    #1. Load HUMAN Data (Label 0) ---
    human_files = glob.glob(os.path.join(base_path, "processed", "*chunked.txt"))

    print(f"Found {len(human_files)} Human files:")
    for file_path in human_files:
        print(f"  - Loading {os.path.basename(file_path)}...")
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            paragraphs = [p.strip() for p in content.split('\n\n') if len(p.split()) > 20]
            for p in paragraphs:
                data.append({"text": p, "label": 0, "source": os.path.basename(file_path)})

    #2. Load AI Data (Label 1) ---
    ai_files = glob.glob(os.path.join(base_path, "generated", "*.txt"))

    print(f"Found {len(ai_files)} AI files:")
    for file_path in ai_files:
        if "_old" in file_path:
            continue

        print(f"  - Loading {os.path.basename(file_path)}...")
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            paragraphs = [p.strip() for p in content.split('\n\n') if len(p.strip()) > 0]
            if len(paragraphs) < 2: 
                paragraphs = [p.strip() for p in content.split('\n') if len(p.strip()) > 20]

            for p in paragraphs:
                data.append({"text": p, "label": 1, "source": os.path.basename(file_path)})

    return pd.DataFrame(data)
df = load_custom_dataset(base_path)

print("\nDataset Balance:")
print(df['label'].value_counts().rename({0: 'Human', 1: 'AI'}))

# Convert to Hugging Face Dataset
from datasets import Dataset
full_dataset = Dataset.from_pandas(df)

# Split
dataset_dict = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_dict['train']
test_dataset = dataset_dict['test']

Found 3 Human files:
  - Loading taleOf2Citieschunked.txt...
  - Loading cthulhuchunked.txt...
  - Loading prideprejudicechunked.txt...
Found 8 AI files:
  - Loading 2cities_class2.txt...
  - Loading cthulhu_class3.txt...
  - Loading prideprejudice_class3.txt...
  - Loading prideprejudice_class2.txt...
  - Loading cthulhu_class2.txt...
  - Loading 2cities_class3.txt...

Dataset Balance:
label
AI       3001
Human    1205
Name: count, dtype: int64


/home/andaburger/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoTokenizer

model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess_function(examples):
    # Truncation=True cuts off texts longer than 256 tokens, padding="max_length" pads shorter ones to 256
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 842/842 [00:00<00:00, 12867.23 examples/s]


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
import numpy as np

# 1. load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    id2label={0: "Human", 1: "AI"},
    label2id={"Human": 0, "AI": 1}
)

# 2. attach peft
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"] # Targeting Attention Heads
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1451.13it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./lora_bert_output",
    eval_strategy="epoch",
    learning_rate=2e-5, 
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Starting Training...")
trainer.train()
trainer.save_model("./final_ghost_detector")
tokenizer.save_pretrained("./final_ghost_detector")

Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.044796,0.990499
2,No log,0.015703,0.996437
3,0.132969,0.013144,0.996437


('./final_ghost_detector/tokenizer_config.json',
 './final_ghost_detector/tokenizer.json')

In [5]:
print("Evaluating on Test Set...")
results = trainer.evaluate()
print(f"Final Accuracy: {results['eval_accuracy']:.4f}")

Evaluating on Test Set...


Final Accuracy: 0.9964
